# Contents
I replace the values in the quantization tables with the nearest power of 2.

This should help the problem of cumulativeness.
Indeed:

`round(round(x / q[t]) * q[t] / q[t+1]) * q[t+1]`

is equivalent to `round(x / q[t+1]) * q[t+1]` just if `q[t+1]` is multiple of `q[t]`

In [ ]:
! pip install jpegio

In [ ]:
from PIL import Image
import numpy as np
import jpegio
import matplotlib.pyplot as plt
import os
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/tesi/" if 'google.colab' in str(get_ipython()) else "."
sample_jpeg_path = os.path.join(BASE_DIR, "experiments/tables/reference_image/000001.jpg")
SAVE_PATH = os.path.join(BASE_DIR, "assets/quant_tables_pow2.pt")

# copmuting power of 2 qt

In [ ]:
# import torch
# import numpy as np

# # Tabella base luminanza JPEG standard (quality 50)
# # È la tabella di riferimento da cui JPEG deriva tutte le altre
# LUMA_BASE = np.array([
#     [16, 11, 10, 16,  24,  40,  51,  61],
#     [12, 12, 14, 19,  26,  58,  60,  55],
#     [14, 13, 16, 24,  40,  57,  69,  56],
#     [14, 17, 22, 29,  51,  87,  80,  62],
#     [18, 22, 37, 56,  68, 109, 103,  77],
#     [24, 35, 55, 64,  81, 104, 113,  92],
#     [49, 64, 78, 87, 103, 121, 120, 101],
#     [72, 92, 95, 98, 112, 100, 103,  99],
# ], dtype=np.float32)

# CHROMA_BASE = np.array([
#     [17, 18, 24, 47, 99, 99, 99, 99],
#     [18, 21, 26, 66, 99, 99, 99, 99],
#     [24, 26, 56, 99, 99, 99, 99, 99],
#     [47, 66, 99, 99, 99, 99, 99, 99],
#     [99, 99, 99, 99, 99, 99, 99, 99],
#     [99, 99, 99, 99, 99, 99, 99, 99],
#     [99, 99, 99, 99, 99, 99, 99, 99],
#     [99, 99, 99, 99, 99, 99, 99, 99],
# ], dtype=np.float32)

# def nearest_pow2(x):
#     # Ensure x is at least 1.0, as quantization tables should always have values >= 1
#     x = x.clamp(min=1.0)
#     # Calculate log2, round to nearest integer, then raise 2 to that power
#     return torch.pow(2.0, torch.round(torch.log2(x)))

# def build_cumulative_quant_tables(time_steps=10, save_path=SAVE_PATH):
#     """
#     Costruisce tabelle cumulative usando potenze di 2.

#     Proprietà garantita: q[t+1] = nearest_pow2(q[t] * 2)
#     → quantizzare con t+1 su immagine già quantizzata con t
#       equivale a quantizzare direttamente con t+1 sull'originale (approssimativamente per potenze di 2).

#     Step 0 → nessuna degradazione (q=nearest_pow2(base))
#     Step 9 → degradazione massima (q=nearest_pow2(q_8 * 2), valori altissimi)
#     """
#     quant_tables = {}

#     # Initialize first step (t=0) by converting LUMA_BASE and CHROMA_BASE to nearest powers of 2
#     prev_luma_tensor = nearest_pow2(torch.tensor(LUMA_BASE, dtype=torch.float32))
#     # prev_chroma_tensor = nearest_pow2(torch.tensor(CHROMA_BASE, dtype=torch.float32))

#     # Force DC (Direct Current) coefficient to 1 for all steps: it should never be quantized
#     prev_luma_tensor[0, 0] = 1.0
#     # prev_chroma_tensor[0, 0] = 1.0
#     prev_chroma_tensor = torch.ones_like(prev_luma_tensor)

#     quant_tables[f'step{0}_luminance']   = prev_luma_tensor
#     quant_tables[f'step{0}_chrominance'] = prev_chroma_tensor

#     # Generate subsequent steps by multiplying the previous step's table by 2 and re-applying nearest_pow2
#     for t in range(1, time_steps):
#         # Calculate next tables by multiplying previous by 2 and then rounding to nearest power of 2
#         luma_tensor = nearest_pow2(prev_luma_tensor * 2.0)
#         # chroma_tensor = nearest_pow2(prev_chroma_tensor * 2.0)

#         # Apply clipping to 32768, as nearest_pow2(value * 2) can exceed it if the base was large
#         luma_tensor = luma_tensor.clamp(max=32768.0)
#         # chroma_tensor = chroma_tensor.clamp(max=32768.0)

#         # Force DC coefficient to 1.0 again, in case rounding affected it
#         luma_tensor[0, 0] = 1.0
#         # chroma_tensor[0, 0] = 1.0

#         quant_tables[f'step{t}_luminance']   = luma_tensor
#         # quant_tables[f'step{t}_chrominance'] = chroma_tensor

#         quant_tables[f'step{t}_chrominance'] = torch.ones_like(luma_tensor)

#         # Update previous tables for the next iteration
#         prev_luma_tensor = luma_tensor
#         # prev_chroma_tensor = chroma_tensor

#         # Verification: Assert monotonicity - tables should always be non-decreasing
#         assert torch.all(luma_tensor >= quant_tables[f'step{t-1}_luminance']), \
#             f"Step {t}: tabella luminanza non monotona!"
#         # assert torch.all(chroma_tensor >= quant_tables[f'step{t-1}_chrominance']), \
#         #     f"Step {t}: tabella crominanza non monotona!"

#     torch.save(quant_tables, save_path)
#     print(f"Salvate {time_steps} tabelle cumulative in '{save_path}'")

#     # Print summary of min/max values for each step for visual inspection
#     print(f"\n{'Step':>4}  {'Luma min':>8}  {'Luma max':>8}  {'Chroma min':>10}  {'Chroma max':>10}")
#     print("-" * 50)
#     for t in range(time_steps):
#         l = quant_tables[f'step{t}_luminance'].numpy()
#         c = quant_tables[f'step{t}_chrominance'].numpy()
#         print(f"{t:>4}  {l.min():>8.0f}  {l.max():>8.0f}  {c.min():>10.0f}  {c.max():>10.0f}")

#     return quant_tables

# quant_tables = build_cumulative_quant_tables(time_steps=10)

The last step consists in keeping only the DC. To do so, I set all the values corresponding to the AC to very high numbers.

But if I do so justy in the last step, then the difference between the ninth and tenth step would be too big. To solve this, I use some steps for transitioning, with the parameter `TRANSITION_STEPS`.

In [ ]:
quant_tables = torch.load(os.path.join(BASE_DIR, 'assets/quant_tables2.pt'))

def nearest_pow2(x):
    x = x.clamp(min=1.0)
    return torch.pow(2.0, torch.round(torch.log2(x)))

quant_tables_pow2 = {}

step_keys = sorted(
    {k.rsplit('_', 1)[0] for k in quant_tables.keys()},
    key=lambda s: int(s.replace('step', ''))
)
n_steps = len(step_keys)

# number of steps used for transitionin to only DC
# e.g., if set to 3 then the last 3 steps grow gradually
TRANSITION_STEPS = 3

for idx, step_name in enumerate(step_keys):
    for suffix in ['luminance']:    #, 'chrominance']:
        key = f'{step_name}_{suffix}'
        value = quant_tables[key]

        steps_from_end = n_steps - 1 - idx  # 0 = last step, 1 = before last, ...

        if steps_from_end == 0:
            # last step: only DC
            t = torch.full_like(value, 1e6)
        elif steps_from_end < TRANSITION_STEPS:
            #transition steps
            base = nearest_pow2(value)
            # aggressive boost
            boost = 2 ** (TRANSITION_STEPS - steps_from_end)
            t = base * boost
        else:
            t = nearest_pow2(value)

        t[0, 0] = 1.0  # preserve DC
        quant_tables_pow2[key] = t

quant_tables_pow2[f'{step_keys[-1]}_luminance'][0, 0] = 1.0
# quant_tables_pow2[f'{step_keys[-1]}_chrominance'][0, 0] = 1.0
# quant_tables_pow2[f'{step_keys[:]}_chrominance'] = torch.ones_like(quant_tables_pow2[f'{step_keys[-1]}_luminance'])

for i in range(n_steps):
    quant_tables_pow2[f'step{i}_chrominance'] = torch.ones_like(quant_tables_pow2[f'step{i}_luminance'])
torch.save(quant_tables_pow2, SAVE_PATH)

print(f"{'Step':>6}  {'Luma max':>10}")
for step_name in step_keys:
    print(f"{step_name:>6}  {quant_tables_pow2[f'{step_name}_luminance'].max().item():>10.0f}")

  Step    Luma max
 step0           1
 step1         256
 step2         256
 step3         256
 step4         256
 step5         256
 step6         256
 step7         512
 step8        1024
 step9     1000000


In [ ]:
for key, value in quant_tables.items():
    if 'luminance' in key:
        print(f"{key}:\n{value}\n")

step0_luminance:
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

step1_luminance:
tensor([[ 40.,  28.,  25.,  40.,  60., 100., 128., 153.],
        [ 30.,  30.,  35.,  48.,  65., 145., 150., 138.],
        [ 35.,  33.,  40.,  60., 100., 143., 173., 140.],
        [ 35.,  43.,  55.,  73., 128., 218., 200., 155.],
        [ 45.,  55.,  93., 140., 170., 255., 255., 193.],
        [ 60.,  88., 138., 160., 203., 255., 255., 230.],
        [123., 160., 195., 218., 255., 255., 255., 253.],
        [180., 230., 238., 245., 255., 250., 255., 248.]])

step2_luminance:
tensor([[ 61.,  42.,  38.,  61.,  92., 154., 196., 234.],
        [ 46.,  46.,  54.,  73., 100., 223., 230., 211.],
        [ 54.,  50.,  61.,  9

In [ ]:
for key, value in quant_tables_pow2.items():
    if 'luminance' in key:
        print(f"{key}:\n{value}\n")

step0_luminance:
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

step1_luminance:
tensor([[  1.,  32.,  32.,  32.,  64., 128., 128., 128.],
        [ 32.,  32.,  32.,  64.,  64., 128., 128., 128.],
        [ 32.,  32.,  32.,  64., 128., 128., 128., 128.],
        [ 32.,  32.,  64.,  64., 128., 256., 256., 128.],
        [ 32.,  64., 128., 128., 128., 256., 256., 256.],
        [ 64.,  64., 128., 128., 256., 256., 256., 256.],
        [128., 128., 256., 256., 256., 256., 256., 256.],
        [128., 256., 256., 256., 256., 256., 256., 256.]])

step2_luminance:
tensor([[  1.,  32.,  32.,  64., 128., 128., 256., 256.],
        [ 64.,  64.,  64.,  64., 128., 256., 256., 256.],
        [ 64.,  64.,  64., 12

In [ ]:
for key, value in quant_tables_pow2.items():
    if 'chrominance' in key:
        print(f"{key}:\n{value}\n")

step0_chrominance:
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

step1_chrominance:
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

step2_chrominance:
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1.

plotting images

In [ ]:
num_images = len(qualities)
plt.figure(figsize=(num_images * 4, 5))

for i in range(num_images):
    # saved_img = jpegio.read(f'output_img_quality_{qualities[i]}.jpg')
    saved_img = Image.open(f'output_img_quality_{i}.jpg').convert('RGB')
    plt.subplot(1, num_images, i + 1)
    plt.imshow(saved_img)
    plt.title(f'step {i}')
    plt.axis('off')

printing quantization tables

In [ ]:
for i in range(num_images):
    saved_img = jpegio.read(f'output_img_quality_{i}.jpg')
    for (quant_table, original_table) in zip(saved_img.quant_tables, jpeg_data.quant_tables):
        print(f"Quality {qualities[i]}:\n{quant_table}")

# saving quantization tables

### pt

In [ ]:
import torch

quant_tables = {}
for i, q in enumerate(qualities):
    saved_img = jpegio.read(f'output_img_quality_{q}.jpg')
    quant_tables[f'step{i}_luminance'] = torch.tensor(
        np.array(saved_img.quant_tables[0], dtype=np.float32)
    )
    quant_tables[f'step{i}_chrominance'] = torch.tensor(
        np.array(saved_img.quant_tables[1], dtype=np.float32)
    )

torch.save(quant_tables, SAVE_PATH)

Verifying that I correctly saved the tables:

In [ ]:
quant_tables = torch.load(SAVE_PATH)

# Accesso diretto, già tensori
lum = quant_tables['step10_luminance']#.to('cuda')
chrom = quant_tables['step10_chrominance']#.to('cuda')

In [ ]:
print(f"Luminance:\n{lum}")
print(f"Chrominance:\n{chrom}")